In [37]:
import pandas as pd
import numpy as np

import os

In [38]:
data = pd.read_csv('HousingData.csv')

In [39]:
data = data.fillna(data.median())

In [58]:
data

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,11.43,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0.0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,11.43,22.4
502,0.04527,0.0,11.93,0.0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0.0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0.0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


1. CRIM
per capita crime rate by town
2. ZN
proportion of residential land zoned for lots over
25,000 sq. ft.
3. INDUS
proportion of non-retail business acres per town
4. CHAS Charles River dummy variable (= 1 if tract bounds
river; 0 otherwise)
5. NOX
nitric oxides concentration (parts per 10 million)
6. RM
average number of rooms per dwelling
7. AGE
proportion of owner-occupied units built prior to 1940 8.
DIS
weighted distances to five Boston employment centres
9. RAD
index of accessibility to radial highways
10. TAX
full-value property-tax rate per $10,000
11. PTRATIO
pupil-teacher ratio by town
12. B
1000 Bk
-0.63)^2 where Bk is the proportion of blacks
by town
13. LSTAT
% lower status of the population
14. HEDV
Median value of owner-occupied homes in $1000's

In [40]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    float64
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    int64  
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  MEDV     506 non-null    float64
dtypes: float64(12), int64(2)
memory usage: 55.5 KB


In [41]:
class MyLineReg:
    def __init__(self, n_iter=500, learning_rate=0.001):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.weights = None  # Новый параметр для хранения весов модели

    def __str__(self):
        return f"MyLineReg class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"

    def fit(self, X, y, verbose=False):
        # Добавляем столбец единиц для учета свободного члена (w0)
        X = np.c_[np.ones((X.shape[0], 1)), X]
        # Определяем количество фичей и создаем вектор весов, состоящий из одних единиц
        self.weights = np.ones(X.shape[1])
        # Градиентный спуск
        for i in range(self.n_iter):
            # Вычисляем предсказания
            y_pred = X.dot(self.weights)
            # Расчет MSE
            mse = np.mean((y - y_pred) ** 2)
            # Вычисляем градиент
            grad = 2 / X.shape[0] * X.T.dot(y_pred - y)
            # Обновляем веса
            self.weights -= self.learning_rate * grad
            # Логирование
            if i % 10 == 0:
                print(f"Итерация {i} | MSE: {mse}. Grad: {np.mean(grad)}")

    def get_coef(self):
        # Возвращаем веса, начиная со второго значения
        return self.weights[1:]
    
    def predict(self, x_pred):
        x_pred=np.c_[np.ones((x_pred.shape[0], 1)),x_pred]
        return x_pred.dot(self.weights)

In [43]:
data_norm = data.copy()

for col in data.columns:
    data_norm[col] = (data[col] - data[col].mean()) / data[col].std()

In [51]:
import random
# генерируем список длиной 152 (70% объема data) и заполняем рандомными int числами диапазона range(len(data)). Это для train
ind_train=random.sample(range(len(data)), k=round(len(data)*0.7))
# используя set работаем со списками как с множествами 
ind_test = set(data.index)  - set(ind_train)

ind_test = list(ind_test)

In [52]:
model = MyLineReg()
X_train = data_norm.drop('MEDV', axis=1).iloc[ind_train].values
X_test = data_norm.drop('MEDV', axis=1).iloc[ind_test].values
y_train = data['MEDV'].iloc[ind_train].values
y_test = data['MEDV'].iloc[ind_test].values

model.fit(X_train,y_train)

Итерация 0 | MSE: 641.2313235570825. Grad: 2.449341926631022
Итерация 10 | MSE: 598.9452018524986. Grad: 1.941175101360171
Итерация 20 | MSE: 562.1679077576076. Grad: 1.4967331692423511
Итерация 30 | MSE: 529.8586115626114. Grad: 1.10858703095554
Итерация 40 | MSE: 501.19510585707184. Grad: 0.7701621094009695
Итерация 50 | MSE: 475.52682884988565. Grad: 0.47564031975596593
Итерация 60 | MSE: 452.3380287027123. Grad: 0.21987327927649605
Итерация 70 | MSE: 431.21887761003103. Grad: -0.0016945317075056008
Итерация 80 | MSE: 411.84281794319594. Grad: -0.19309379844101457
Итерация 90 | MSE: 393.94879400284105. Grad: -0.35789060008845147
Итерация 100 | MSE: 377.32731391677515. Grad: -0.49923973203295
Итерация 110 | MSE: 361.8095143205041. Grad: -0.6199319138714469
Итерация 120 | MSE: 347.2585792580778. Grad: -0.7224355840812214
Итерация 130 | MSE: 333.56300489737845. Grad: -0.808933901973413
Итерация 140 | MSE: 320.6313115185929. Grad: -0.881357506401298
Итерация 150 | MSE: 308.3878903545347

In [62]:
model.get_coef()

array([-0.25265149,  1.07083237, -0.59073804,  1.41627235,  0.27767349,
        3.51918949,  0.22257261,  0.16638553,  0.02954361, -0.63962315,
       -1.20485931,  1.30386528, -1.79653767])